In [11]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted!


In [3]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

filepath = "/content/drive/MyDrive/Colab Notebooks/output.jsonl"

data = []
with open(filepath, "r") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total loaded: {len(df)}")
print(df['label'].value_counts())

# Sample 5000: 70% benign (3500), 30% phishing (1500)
N_BENIGN = 3500
N_PHISHING = 1500

benign_df = df[df['label'] == 'benign'].sample(n=N_BENIGN, random_state=42)
phishing_df = df[df['label'] == 'phishing'].sample(n=N_PHISHING, random_state=42)
subset = pd.concat([benign_df, phishing_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nSubset: {len(subset)}")
print(subset['label'].value_counts())

train_df, test_df = train_test_split(subset, test_size=0.2, stratify=subset['label'], random_state=42)
print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")
print(f"Train distribution:\n{train_df['label'].value_counts()}")
print(f"Test distribution:\n{test_df['label'].value_counts()}")

Total loaded: 123380
label
phishing    77600
benign      45780
Name: count, dtype: int64

Subset: 5000
label
benign      3500
phishing    1500
Name: count, dtype: int64

Train: 4000 | Test: 1000
Train distribution:
label
benign      2800
phishing    1200
Name: count, dtype: int64
Test distribution:
label
benign      700
phishing    300
Name: count, dtype: int64


In [4]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [7]:
import json
import pandas as pd

# Load with error handling for malformed lines
data = []
errors = []
with open(filepath, "r") as f:
    for i, line in enumerate(f):
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            errors.append(i)

print(f"Successfully parsed: {len(data)}")
print(f"Failed lines: {len(errors)}")
if errors:
    print(f"Failed line indices (first 10): {errors[:10]}")

# Convert to DataFrame
df = pd.DataFrame(data)

# --- Overview ---
print(f"\n{'='*50}")
print(f"DATASET OVERVIEW")
print(f"{'='*50}")
print(f"Total sites: {len(df)}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

# --- Label distribution ---
print(f"\n{'='*50}")
print(f"LABEL DISTRIBUTION")
print(f"{'='*50}")
print(df["label"].value_counts())
print(f"\n{df['label'].value_counts(normalize=True).map('{:.2%}'.format)}")

# --- Column types and sample values ---
print(f"\n{'='*50}")
print(f"COLUMN DETAILS")
print(f"{'='*50}")
for col in df.columns:
    sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
    dtype = type(sample).__name__
    nulls = df[col].isnull().sum()
    print(f"\n[{col}]")
    print(f"  Type: {dtype} | Nulls: {nulls}")
    print(f"  Sample: {str(sample)[:150]}")

Successfully parsed: 123380
Failed lines: 0

DATASET OVERVIEW
Total sites: 123380
Columns (7): ['label', 'url', 'hostname', 'registered_domain', 'title', 'visible_text_excerpt', 'families']

LABEL DISTRIBUTION
label
phishing    77600
benign      45780
Name: count, dtype: int64

label
phishing    62.90%
benign      37.10%
Name: proportion, dtype: object

COLUMN DETAILS

[label]
  Type: str | Nulls: 0
  Sample: phishing

[url]
  Type: str | Nulls: 0
  Sample: https://app46657.swiftway.in/webserver103/?8788898=3mail@a.b.c0

[hostname]
  Type: str | Nulls: 0
  Sample: app46657.swiftway.in

[registered_domain]
  Type: str | Nulls: 0
  Sample: swiftway.in

[title]
  Type: str | Nulls: 0
  Sample: Outlook Web App

[visible_text_excerpt]
  Type: str | Nulls: 0
  Sample: Outlook Web App Webmail Web mail access application Security( show explanation ) This is a public or shared computer This is a private computer Use th

[families]
  Type: dict | Nulls: 0
  Sample: {'domain': [{'id': 'domain.cla

In [9]:
data = []
with open(filepath, "r") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total loaded: {len(df)}")

# We want 5000 samples: 70% benign (3500), 30% phishing (1500)
N_BENIGN = 3500
N_PHISHING = 1500

benign_df = df[df['label'] == 'benign'].sample(n=N_BENIGN, random_state=42)
phishing_df = df[df['label'] == 'phishing'].sample(n=N_PHISHING, random_state=42)
subset = pd.concat([benign_df, phishing_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nSubset: {len(subset)}")
print(subset['label'].value_counts())

# Split: 80% train, 20% test (stratified)
train_df, test_df = train_test_split(subset, test_size=0.2, stratify=subset['label'], random_state=42)
print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")
print(f"Train distribution:\n{train_df['label'].value_counts()}")
print(f"Test distribution:\n{test_df['label'].value_counts()}")

Total loaded: 123380

Subset: 5000
label
benign      3500
phishing    1500
Name: count, dtype: int64

Train: 4000 | Test: 1000
Train distribution:
label
benign      2800
phishing    1200
Name: count, dtype: int64
Test distribution:
label
benign      700
phishing    300
Name: count, dtype: int64


In [12]:
import torch
import gc
import time
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ---- System prompt ----
SYSTEM_PROMPT = "You are a cybersecurity analyst. Analyze the website data and respond with exactly one word: phishing or benign."

# ---- Format a row into a user prompt ----
def website_to_prompt(row):
    """Convert a dataframe row into a structured prompt."""
    # Extract family signals if available
    families_str = ""
    if pd.notna(row.get('families')) and isinstance(row['families'], dict):
        signals = []
        for category, items in row['families'].items():
            if isinstance(items, list):
                for item in items[:3]:  # limit to 3 signals per category
                    if isinstance(item, dict):
                        sig_id = item.get('id', 'unknown')
                        direction = item.get('direction', '')
                        signals.append(f"{sig_id} ({direction})")
        if signals:
            families_str = "\nDetected signals: " + "; ".join(signals[:8])  # cap at 8 total

    title = row.get('title', 'N/A') or 'N/A'
    text_excerpt = row.get('visible_text_excerpt', 'N/A') or 'N/A'
    if isinstance(text_excerpt, str) and len(text_excerpt) > 300:
        text_excerpt = text_excerpt[:300] + "..."

    prompt = f"""URL: {row.get('url', 'N/A')}
Hostname: {row.get('hostname', 'N/A')}
Registered domain: {row.get('registered_domain', 'N/A')}
Page title: {title}
Visible text: {text_excerpt}{families_str}

Is this website phishing or benign?"""
    return prompt

# ---- Model loading ----
def load_model(model_name, use_4bit=True):
    """Load model with optional 4-bit quantization."""
    print(f"\n{'='*60}")
    print(f"Loading: {model_name} ({'4-bit' if use_4bit else 'bf16'})")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_config,
            device_map="auto", trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=torch.bfloat16,
            device_map="auto", trust_remote_code=True
        )

    print(f"✅ Loaded! Params: {model.num_parameters():,}")
    return model, tokenizer

# ---- Inference ----
def ask(model, tokenizer, user_prompt):
    """Get model response for a single prompt."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return response

# ---- Evaluation ----
def parse_label(response):
    """Extract phishing/benign from model response."""
    resp = response.lower().strip()
    if 'phishing' in resp:
        return 'phishing'
    elif 'benign' in resp or 'legitimate' in resp or 'safe' in resp:
        return 'benign'
    return None

ALL_RESULTS = {}

def evaluate_model(model, tokenizer, model_name, eval_df, n_samples=200):
    """Evaluate model on n_samples from eval_df."""
    print(f"\n📊 Evaluating: {model_name} ({n_samples} samples)...")
    sample = eval_df.sample(n=min(n_samples, len(eval_df)), random_state=42)

    y_true, y_pred = [], []
    failures = 0

    for i, (_, row) in enumerate(sample.iterrows()):
        if (i + 1) % 50 == 1:
            print(f"  {i+1}/{len(sample)}...")

        prompt = website_to_prompt(row)
        response = ask(model, tokenizer, prompt)
        pred = parse_label(response)

        if pred is None:
            failures += 1
            pred = 'phishing'  # default for unparseable

        y_true.append(row['label'])
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label='phishing', zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label='phishing', zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label='phishing', zero_division=0)

    print(f"  Accuracy: {acc:.3f} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f} | Failures: {failures}")

    ALL_RESULTS[model_name] = {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'y_true': y_true, 'y_pred': y_pred, 'failures': failures
    }
    return acc, f1

# ---- Training data preparation ----
def prepare_sft_dataset(tokenizer, df):
    """Format training data for SFTTrainer."""
    formatted = []
    for _, row in df.iterrows():
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": website_to_prompt(row)},
            {"role": "assistant", "content": row['label']},  # "phishing" or "benign"
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

# ---- LoRA setup ----
def setup_lora(model):
    """Apply LoRA adapter."""
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=16, lora_alpha=16, lora_dropout=0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    trainable, total = model.get_nb_trainable_parameters()
    print(f"  LoRA: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model

# ---- Fine-tuning ----
def fine_tune(model, tokenizer, train_dataset, output_dir, epochs=3):
    """Run SFT training."""
    split = train_dataset.train_test_split(test_size=0.1, seed=42)

    args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        optim="adamw_8bit",
    )

    trainer = SFTTrainer(
        model=model, args=args,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        processing_class=tokenizer,
    )

    print(f"  Training {len(split['train'])} examples, {epochs} epochs...")
    start = time.time()
    trainer.train()
    duration = time.time() - start
    print(f"  ✅ Done in {duration/60:.1f} min")
    return duration

# ---- Cleanup ----
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU cleared.")

print("✅ All helper functions loaded!")

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
cannot import name 'deepspeed_sp_compute_loss' from 'transformers.integrations.deepspeed' (/usr/local/lib/python3.12/dist-packages/transformers/integrations/deepspeed.py)

In [ ]:
# For 5000 data, test set is 1000. Evaluating all 1000 takes a while.
# Let's use 200 for quick results. You can increase later.
NUM_TEST_SAMPLES = 200
print(f"Will evaluate on {NUM_TEST_SAMPLES} test samples per model.")
print(f"Total test set available: {len(test_df)}")

In [ ]:
MODEL_1 = "mistralai/Mistral-7B-Instruct-v0.3"
model1, tok1 = load_model(MODEL_1, use_4bit=True)

# Baseline evaluation
acc1_base, f1_1_base = evaluate_model(model1, tok1, "Mistral-7B (baseline)", test_df, NUM_TEST_SAMPLES)

# Show sample responses
print("\nSample responses:")
for label_val in ['phishing', 'benign']:
    sample = test_df[test_df['label'] == label_val].iloc[0]
    resp = ask(model1, tok1, website_to_prompt(sample))
    print(f"  [{label_val.upper()}] → Model says: '{resp}'")

In [ ]:
# Prepare data and fine-tune
dataset1 = prepare_sft_dataset(tok1, train_df)
model1 = setup_lora(model1)
time1 = fine_tune(model1, tok1, dataset1, "./mistral-7b-website")

# Evaluate after fine-tuning
acc1_ft, f1_1_ft = evaluate_model(model1, tok1, "Mistral-7B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

print("\nFine-tuned sample responses:")
for label_val in ['phishing', 'benign']:
    sample = test_df[test_df['label'] == label_val].iloc[0]
    resp = ask(model1, tok1, website_to_prompt(sample))
    print(f"  [{label_val.upper()}] → Model says: '{resp}'")

# Free memory
del model1, tok1, dataset1
cleanup()

In [ ]:
# TODO: Add your Hugging Face login here before running this notebook.
# from huggingface_hub import login
# login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

In [ ]:
MODEL_2 = "meta-llama/Llama-3.2-3B-Instruct"
model2, tok2 = load_model(MODEL_2, use_4bit=True)

# Baseline evaluation
acc2_base, f1_2_base = evaluate_model(model2, tok2, "Llama-3.2-3B (baseline)", test_df, NUM_TEST_SAMPLES)

# Show sample responses
print("\nSample responses:")
for label_val in ['phishing', 'benign']:
    sample = test_df[test_df['label'] == label_val].iloc[0]
    resp = ask(model2, tok2, website_to_prompt(sample))
    print(f"  [{label_val.upper()}] → Model says: '{resp}'")

In [ ]:
# Prepare data and fine-tune
dataset2 = prepare_sft_dataset(tok2, train_df)
model2 = setup_lora(model2)
time2 = fine_tune(model2, tok2, dataset2, "./llama-3b-website")

# Evaluate after fine-tuning
acc2_ft, f1_2_ft = evaluate_model(model2, tok2, "Llama-3.2-3B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

print("\nFine-tuned sample responses:")
for label_val in ['phishing', 'benign']:
    sample = test_df[test_df['label'] == label_val].iloc[0]
    resp = ask(model2, tok2, website_to_prompt(sample))
    print(f"  [{label_val.upper()}] → Model says: '{resp}'")

del model2, tok2, dataset2
cleanup()

In [ ]:
print("=" * 85)
print("📊 MULTI-MODEL COMPARISON — PHISHING WEBSITE DETECTION")
print("=" * 85)
print(f"Dataset: output.jsonl | 5000 samples (70% benign, 30% phishing)")
print(f"Train: {len(train_df)} | Test eval: {NUM_TEST_SAMPLES}")
print(f"Method: LoRA r=16 alpha=16 | lr=2e-4 | 3 epochs | adamw_8bit")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()

models = [
    ("Mistral-7B", "Mistral-7B (baseline)", "Mistral-7B (fine-tuned)"),
    ("Llama-3.2-3B", "Llama-3.2-3B (baseline)", "Llama-3.2-3B (fine-tuned)"),
]

print(f"{'Model':<16} │ {'--- Baseline ---':^27} │ {'--- Fine-tuned ---':^27} │ {'ΔF1':^7}")
print(f"{'':16} │ {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} │ {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} │")
print("─" * 85)

for name, bk, fk in models:
    if bk in ALL_RESULTS and fk in ALL_RESULTS:
        b = ALL_RESULTS[bk]
        f = ALL_RESULTS[fk]
        d = f['f1'] - b['f1']
        ds = f"+{d:.2f}" if d >= 0 else f"{d:.2f}"
        print(f"{name:<16} │ {b['accuracy']:.3f} {b['precision']:.3f} {b['recall']:.3f} {b['f1']:.3f} │ {f['accuracy']:.3f} {f['precision']:.3f} {f['recall']:.3f} {f['f1']:.3f} │ {ds:>6}")
    elif bk in ALL_RESULTS:
        b = ALL_RESULTS[bk]
        print(f"{name:<16} │ {b['accuracy']:.3f} {b['precision']:.3f} {b['recall']:.3f} {b['f1']:.3f} │ {'(not run)':^27} │")

print("─" * 85)

# Detailed reports
print("\n📋 DETAILED CLASSIFICATION REPORTS:")
for name, bk, fk in models:
    for key in [bk, fk]:
        if key in ALL_RESULTS:
            r = ALL_RESULTS[key]
            print(f"\n--- {key} ---")
            print(classification_report(r['y_true'], r['y_pred'],
                  target_names=['Benign', 'Phishing'], digits=3, zero_division=0))

print("\n🏆 KEY FINDINGS:")
for name, bk, fk in models:
    if bk in ALL_RESULTS and fk in ALL_RESULTS:
        d = ALL_RESULTS[fk]['f1'] - ALL_RESULTS[bk]['f1']
        status = "✅ IMPROVED" if d > 0.01 else "⚠️ WORSE" if d < -0.01 else "➡️ SAME"
        print(f"  {name}: {status} (F1: {ALL_RESULTS[bk]['f1']:.3f} → {ALL_RESULTS[fk]['f1']:.3f}, Δ={d:+.3f})")